In [ ]:
from google.colab import userdata
import urllib.parse
from sqlalchemy import create_engine

try:
    # Traemos los secretos
    raw_pass = userdata.get('SUPABASE_PASSWORD')
    user = userdata.get('SUPABASE_USER')
    host = userdata.get('SUPABASE_HOST')
    port = userdata.get('SUPABASE_PORT')
    dbname = userdata.get('SUPABASE_DBNAME')
    
    # Escapamos la contraseña
    password = urllib.parse.quote_plus(raw_pass)
    
    # Intentamos conectar
    db_url = f"postgresql://{user}:{password}@{host}:{port}/{dbname}?sslmode=require"
    engine = create_engine(db_url)
    
    with engine.connect() as conn:
        print("✅ ¡Conexión exitosa! Las llaves de Colab están funcionando perfecto.")
        
except Exception as e:
    print(f"❌ Error: {e}")
    print("\n💡 Tip: Revisa que el nombre en el panel de la llave sea EXACTAMENTE igual al del código.")

✅ ¡CONEXIÓN EXITOSA! El túnel IPv4 del Pooler está funcionando.


In [ ]:
#Carga de data en Bronce

import pandas as pd
import os
from sqlalchemy import text
from google.colab import drive 

# 1. Montar Drive (si no está montado)
drive.mount('/content/drive')

# 2. Ruta de tus archivos
PATH_DRIVE = '/content/drive/MyDrive/SEMESTRE 4/Proyecto productivo IIB/IDL01/Bronce_Data raw /'

def ejecutar_pipeline_bronce():
    try:
        # Usamos text() también aquí para la consulta inicial
        query_config = text("SELECT nombre_tabla FROM meta.pipeline_config WHERE estado = 'ACTIVE'")
        tablas_activas = pd.read_sql(query_config, engine)['nombre_tabla'].tolist()
        print(f"📋 Tablas encontradas en metadatos: {len(tablas_activas)}")
    except Exception as e:
        print(f"❌ Error al leer metadatos: {e}")
        return

    for tabla in tablas_activas:
        # Usamos la ruta con el espacio al final que descubrimos antes
        archivo_path = os.path.join(PATH_DRIVE, f"{tabla}.csv")
        
        if os.path.exists(archivo_path):
            print(f"⌛ Subiendo {tabla}...")
            df = pd.read_csv(archivo_path)
            df.columns = [c.lower().replace(' ', '_').replace('.', '_') for c in df.columns]
            df['etl_fecha_carga'] = pd.Timestamp.now()
            
            # Cargar datos
            df.to_sql(tabla, engine, schema='bronce', if_exists='replace', index=False)
            
            # --- CAMBIO AQUÍ: Usar text() y commit() ---
            with engine.connect() as conn:
                # Envolvemos el UPDATE en text()
                statement = text(f"UPDATE meta.pipeline_config SET ultima_carga = NOW() WHERE nombre_tabla = '{tabla}'")
                conn.execute(statement)
                conn.commit() # Importante para que el Pooler guarde los cambios
            
            print(f"   ✅ {tabla} lista en Bronce.")
        else:
            print(f"   ⚠️ No se encontró el archivo: {archivo_path}")

# Ejecutar
ejecutar_pipeline_bronce()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📋 Tablas encontradas en metadatos: 9
⌛ Subiendo ads_campanas_maestro...
   ✅ ads_campanas_maestro lista en Bronce.
⌛ Subiendo ads_insights_diario...
   ✅ ads_insights_diario lista en Bronce.
⌛ Subiendo clima_diario_log...
   ✅ clima_diario_log lista en Bronce.
⌛ Subiendo sap_canales_maestro...
   ✅ sap_canales_maestro lista en Bronce.
⌛ Subiendo sap_clientes_maestro...
   ✅ sap_clientes_maestro lista en Bronce.
⌛ Subiendo sap_inventario_diario...
   ✅ sap_inventario_diario lista en Bronce.
⌛ Subiendo sap_productos_maestro...
   ✅ sap_productos_maestro lista en Bronce.
⌛ Subiendo sap_ventas_cabecera...
   ✅ sap_ventas_cabecera lista en Bronce.
⌛ Subiendo sap_ventas_detalle...
   ✅ sap_ventas_detalle lista en Bronce.
